# Project 3 — Early Warning System for Sepsis (PhysioNet 2019)
## Notebook 02 — Leakage-safe Temporal Feature Engineering (Windowed Vitals)

**Goal**
Turn the long ICU time-series table into a model-ready feature matrix with **no label leakage**.

**Inputs (from Notebook 01)**
- `results/sample_long_table.parquet` — pre-onset rows only + `y_within_H`

**Outputs**
- `results/features_02b.parquet` — engineered features + identifiers + target
- `results/feature_manifest.json` — feature list & configuration

**Leakage rule (non-negotiable)**
All features at time `t` must be computed using data from times **< t** only.
We enforce this using a **shift(1)** within each patient before applying rolling windows.


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)

# -------------------------
# Paths (EDIT THIS ONCE)
# -------------------------
PROJECT_ROOT = Path(".").resolve()
DATA_ROOT = PROJECT_ROOT  # set this if your dataset/results live elsewhere

RESULTS_DIR = DATA_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IN_LONG = RESULTS_DIR / "sample_long_table.parquet"
OUT_FEATS = RESULTS_DIR / "features_02b.parquet"
OUT_MANIFEST = RESULTS_DIR / "feature_manifest.json"

print("Input:", IN_LONG, "exists:", IN_LONG.exists())


### 1) Load long table
This should already be **pre-onset** rows only (Notebook 01 censored), and include:
- `patient_id`, `ICULOS`, `y_within_H`


In [ ]:
df = pd.read_parquet(IN_LONG)
df = df.sort_values(["patient_id", "ICULOS"]).reset_index(drop=True)

required = {"patient_id", "ICULOS", "y_within_H"}
missing_req = required - set(df.columns)
assert not missing_req, f"Missing required columns: {missing_req}"

print("Rows:", len(df), "Patients:", df["patient_id"].nunique(), "Cols:", df.shape[1])
df.head()


### 2) Choose raw features to engineer
We exclude identifiers and label columns. Everything else becomes a candidate raw feature.


In [ ]:
ID_COLS = ["patient_id", "ICULOS"]
LABEL_COLS = ["y_within_H"]  # SepsisLabel/event_iculos were removed in Notebook 01 output

raw_cols = [c for c in df.columns if c not in ID_COLS + LABEL_COLS]
print("Raw candidate columns:", len(raw_cols))
raw_cols[:20]


### 3) Optional: within-patient forward fill
ICU time-series are very sparse. Forward filling often helps, **but** we preserve missingness signals via flags.

- `ffill_within_patient=True`: forward fill each raw variable within each patient.
- Missingness flags are computed from the **original** missingness (before fill).


In [ ]:
ffill_within_patient = True

df_raw = df[ID_COLS + LABEL_COLS + raw_cols].copy()

# Missingness flags before fill (strong predictors)
for c in raw_cols:
    df_raw[f"{c}__miss"] = df_raw[c].isna().astype(np.int8)

if ffill_within_patient:
    df_raw[raw_cols] = (
        df_raw.groupby("patient_id", sort=False)[raw_cols]
              .ffill()
    )

df_raw.head()


### 4) Leakage-safe temporal features (shift(1) + rolling windows)

We compute features using:
1) `shift(1)` inside each patient so current hour values don't leak into features
2) rolling windows on the shifted values:
   - mean, min, max, std over windows `[6, 12, 24]`
3) short-term change features:
   - `delta_1h` = x(t-1) - x(t-2)  (implemented on shifted series)
   - `delta_6h` = x(t-1) - x(t-7)

All of these use **past-only** information.


In [ ]:
WINDOWS = [6, 12, 24]

def add_temporal_features(g: pd.DataFrame, cols: list[str], windows: list[int]) -> pd.DataFrame:
    g = g.sort_values("ICULOS").copy()

    # Past-only base series
    shifted = g[cols].shift(1)

    out = pd.DataFrame(index=g.index)

    # Rolling stats
    for w in windows:
        roll = shifted.rolling(window=w, min_periods=1)
        out[[f"{c}__mean_{w}" for c in cols]] = roll.mean().to_numpy()
        out[[f"{c}__min_{w}" for c in cols]]  = roll.min().to_numpy()
        out[[f"{c}__max_{w}" for c in cols]]  = roll.max().to_numpy()
        out[[f"{c}__std_{w}" for c in cols]]  = roll.std(ddof=0).to_numpy()

    # Deltas (past-only)
    out[[f"{c}__delta_1h" for c in cols]] = (shifted - shifted.shift(1)).to_numpy()
    out[[f"{c}__delta_6h" for c in cols]] = (shifted - shifted.shift(6)).to_numpy()

    # Last observed (t-1)
    out[[f"{c}__prev1" for c in cols]] = shifted.to_numpy()

    return out

# Only engineer numeric columns (most are numeric in this dataset)
num_cols = [c for c in raw_cols if pd.api.types.is_numeric_dtype(df_raw[c])]
print("Numeric raw columns:", len(num_cols))

feat_blocks = []
for pid, g in df_raw.groupby("patient_id", sort=False):
    feats_g = add_temporal_features(g, num_cols, WINDOWS)
    feat_blocks.append(feats_g)

X_time = pd.concat(feat_blocks).sort_index()

print("Temporal feature matrix shape:", X_time.shape)
X_time.head()


### 5) Combine identifiers + target + engineered features + missingness flags
We keep `patient_id` and `ICULOS` for grouped CV and for early-warning policy later.


In [ ]:
# Collect miss flags
miss_cols = [c for c in df_raw.columns if c.endswith("__miss")]

out = pd.concat(
    [
        df_raw[ID_COLS + LABEL_COLS + miss_cols].reset_index(drop=True),
        X_time.reset_index(drop=True),
    ],
    axis=1
)

# Drop the first ICU hour per patient if desired (most features are NaN-ish there)
# We keep it, but you can drop it for modeling:
drop_first_hour = True
if drop_first_hour:
    first_mask = out.groupby("patient_id")["ICULOS"].transform("min") == out["ICULOS"]
    out = out.loc[~first_mask].reset_index(drop=True)

print("Final feature table:", out.shape)
out.head()


### 6) Leakage sanity checks
We ensure:
- No raw (current-time) columns are accidentally included
- Feature columns follow expected naming
- No patient_id mixing


In [ ]:
# Ensure raw cols are NOT present (we only want engineered + miss flags)
present_raw = [c for c in raw_cols if c in out.columns]
assert len(present_raw) == 0, f"Raw columns leaked into output: {present_raw[:10]}"

# Ensure label + IDs present
for c in ID_COLS + LABEL_COLS:
    assert c in out.columns, f"Missing column: {c}"

# Quick check: no duplicate rows for same (patient_id, ICULOS)
dups = out.duplicated(subset=["patient_id", "ICULOS"]).sum()
assert dups == 0, f"Duplicate (patient_id, ICULOS) rows: {dups}"

print("Leakage checks passed ✅")


### 7) Save artifacts
We save parquet + a manifest describing:
- windows used
- ffill setting
- feature column names


In [ ]:
out.to_parquet(OUT_FEATS, index=False)

feature_cols = [c for c in out.columns if c not in ID_COLS + LABEL_COLS]
manifest = {
    "created_from": str(IN_LONG),
    "windows": WINDOWS,
    "ffill_within_patient": bool(ffill_within_patient),
    "drop_first_hour": bool(drop_first_hour),
    "n_rows": int(out.shape[0]),
    "n_patients": int(out["patient_id"].nunique()),
    "n_features": int(len(feature_cols)),
    "id_cols": ID_COLS,
    "label_cols": LABEL_COLS,
    "feature_cols_preview": feature_cols[:50],
}

OUT_MANIFEST.write_text(json.dumps(manifest, indent=2))
print("Saved:")
print(" -", OUT_FEATS)
print(" -", OUT_MANIFEST)
